# Scraping WDumper data

<a href="https://colab.research.google.com/github/itamargiv/wdumper-scraper/blob/main/notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In order to better understand what kinds of entity data dump subsets our users are interested in, this repository scrapes all dump subsets listed under "recent dumps". The scrape includes a JSON representation of the filters that were used to generate the dump.

The notebook generates a csv file that includes filter data in a human-readable form. Each row of the csv includes the following columns:
- dump name
- URL
- filter (in human-readable form including labels for any items and properties used)
- statements included in the dump (in human-readable form)
- labels (yes/no)
- descriptions (yes/no)
- aliases (yes/no)
- sitelinks (yes/no)
- languages

## Define Utility Functions

The following code block (hidden by default) defines a utility class `GitMetadata` that extracts metadata about the Git repository from either the Colab environment or the local git repository. This metadata includes the username, repository name, branch, path, and URL. The class provides two class methods, `from_colab` and `from_local`, to handle both environments seamlessly.

In [ ]:
from __future__ import annotations

import os
import re
import subprocess

from typing import NamedTuple

class GitMetadata(NamedTuple):
    username: str
    repo: str
    branch: str
    path: str
    url: str

    @classmethod
    def from_colab(cls) -> GitMetadata:
        notebook_url = os.environ.get('COLAB_NOTEBOOK_ID', '')

        if not notebook_url:
            raise ValueError("COLAB_NOTEBOOK_ID environment variable not set")

        pattern = (
            r'github\.com/(?P<user>[^/]+)/'
            r'(?P<repo>[^/]+)/blob/'
            r'(?P<branch>[^/]+)/'
        )
        match = re.search(pattern, notebook_url)

        if not match:
            raise ValueError("URL format not recognized as GitHub")

        groups = match.groupdict()
        url = f"https://github.com/{groups['user']}/{groups['repo']}.git"
        path = f"/content/{groups['repo']}"

        return cls(
            username=groups['user'],
            repo=groups['repo'],
            branch=groups['branch'],
            path=path,
            url=url
        )

    @classmethod
    def from_local(cls) -> GitMetadata:
        def git(*args):
            return subprocess.check_output(['git', *args], text=True).strip()

        remote_url = git('remote', 'get-url', 'origin')
        repo = os.path.basename(git('rev-parse', '--show-toplevel'))
        branch = git('rev-parse', '--abbrev-ref', 'HEAD')

        # username still requires parsing the remote URL
        match = re.search(r'github\.com[:/](?P<username>[^/]+)/', remote_url)
        if not match:
            raise ValueError(f"Remote URL not recognized as GitHub: {remote_url}")

        username = match.group('username')
        url = f"https://github.com/{username}/{repo}"
        path = "."

        return cls(username=username, repo=repo, branch=branch, path=path, url=url)

In [ ]:
import argparse
from IPython.core.magic import magics_class, line_cell_magic, Magics
from IPython.utils.capture import capture_output

def _make_parser():
    parser = argparse.ArgumentParser(add_help=False)
    parser.add_argument("--quiet", "-q", action="store_true")
    return parser

_parser = _make_parser()

@magics_class
class ConditionalMagics(Magics):

    def _run(self, cell: str, quiet: bool) -> None:
        if quiet:
            with capture_output(stdout=True, stderr=False, display=False):
                result = self.shell.run_cell(cell)
        else:
            result = self.shell.run_cell(cell)
        if result.error_in_exec is not None:
            raise result.error_in_exec

    @staticmethod
    def _parse(line: str):
        """Returns (condition_str, quiet flag)."""
        args, remaining = _parser.parse_known_args(line.split())
        return " ".join(remaining), args.quiet

    @line_cell_magic
    def when(self, line: str, cell: str | None = None) -> None:
        condition_str, quiet = self._parse(line)
        if not eval(condition_str, self.shell.user_ns): return
        if cell: self._run(cell, quiet)

    @line_cell_magic
    def unless(self, line: str, cell: str | None = None) -> None:
        condition_str, quiet = self._parse(line)
        if eval(condition_str, self.shell.user_ns): return
        if cell: self._run(cell, quiet)


get_ipython().register_magics(ConditionalMagics)

## Setup Environment

This notebook can be run on Google  (Colab). To set up the environment in Colab, we need to clone the repository and install the required dependencies. This may take a few seconds.

In [ ]:
import sys

IN_COLAB = "google.colab" in sys.modules
REPO = GitMetadata.from_colab() if IN_COLAB else GitMetadata.from_local()

In [ ]:
%%when IN_COLAB --quiet

!rm -rf {REPO.path}

!apt-get install -y git-lfs
!git lfs install
!git clone --branch {REPO.branch} --depth 1 {REPO.url}
!pip install {REPO.path}

## Scrape Dump Data

The code snippet below scrapes all dumps listed under "recent dumps". It uses a thread pool to speed up the scraping process, and it handles any exceptions that may occur during scraping. The scraped data is stored in a list of dictionaries, which can then be converted to a csv file for further analysis.

In [ ]:
from wdumper_scraper import WDumperScraper

results = WDumperScraper(
    cache_path=REPO.path,
    max_workers=10,
    max_retries=3
).run()

## Display Scrape Results

The following code snippet converts the scraped data into a pandas DataFrame and displays it. Each row of the DataFrame corresponds to a dump, and the columns include the dump name, URL, filter, statements included in the dump, and whether labels, descriptions, aliases, sitelinks, and languages are included in the dump.

In [ ]:
from pandas import DataFrame

df = DataFrame(results.dumps, columns=[
    "id",
    "name",
    "labels",
    "descriptions",
    "aliases",
    "sitelinks",
    "languages",
    "url"
])

df

## Save Results to CSV

The code block below saves the DataFrame to a CSV file. The file is named with the current date and saved in the "data" directory of the repository. If the notebook is being run in Colab, a download button is displayed to allow the user to download the CSV file directly.

In [ ]:
from datetime import date

file_path = f"{REPO.path}/data/{date.today()}_wdumper_dumps.csv"
df.to_csv(file_path, index=False)

In [ ]:
%%when IN_COLAB
import ipywidgets as widgets
from IPython.display import display
import google

button = widgets.Button(description="Download CSV", button_style="success")
button.on_click(lambda _: google.colab.files.download(file_path))
display(button)

In [ ]:
%%unless IN_COLAB
print(f"CSV file saved to {file_path}.")